# Importing libraries

In [ ]:
import foldersfiles as ff
import functions as fn

import numpy as np
import pandas as pd
import math

import json

from pathlib import Path

import pickle

# Importing data

In [ ]:
df = fn.import_func(
    ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS / f'{ff.EUMACO}-all.parquet'
)

Drop `json_id` and `transm_id` columns

In [ ]:
# df = df[[
#     'pseudopaz' ,
#     # 'json_id' ,
#     # 'transm_id' ,
#     'datetime' ,
#     'syst_bp_mmhg' ,
#     'diast_bp_mmhg' ,
#     'map_mmhg' ,
#     'hr_bpm' ,
#     'hr_source' ,
#     'spo2_pct' ,
#     'weight_kg' ,
# ]]

# Analysis

In [ ]:
df.info()

## Conflicts and merge candidates

In [ ]:
def check_measurements(df, thresholds):
    """
    thresholds:
        {measurement_column: (maximum_time_difference_minutes,
                              maximum_acceptable_value_difference)}

    Returns all pairs within the time threshold, classified by value difference.
    Missing measurements are ignored. The input dataframe is not modified.

    ag8l
    STARTS by checking if two values stays within the same TIME threshold.
    If yes, checks the value delta:
        - if value delta is greater than the value threshold, marks as a conflict
        - if value delta is lesser than the value threshold, marks as a merge candidate

    In case of more than 3 valid rows, it doesn't show a single result for the whole group;
      instead it shows an entry for all of the couples, like triangular numbers:
      -> if n=number of duplicated rows, it shows ((n*(n-1))/2) entries

    TODO another version that shows a single result for a whole group within the threshold;
      it's unclear if, in case of entries like 10:10, 10:15, 10:20, it should:
      - allow overlapping groups
      - allow groups of extended timespan
    """
    columns = [
        "pseudopaz", "json_id", "transm_id", "measurement", "values",
        "status", "datetimes", "time_delta", "value_delta", "row_indices",
    ]
    results = []

    def unique_ids(a, b):
        return (a,) if a == b else (a, b)

    for measurement, (time_limit, value_limit) in thresholds.items():
        time_limit = pd.Timedelta(minutes=time_limit)

        if pd.isna(time_limit) or time_limit < pd.Timedelta(0):
            raise ValueError(f"Invalid time threshold for {measurement}")
        if pd.isna(value_limit) or value_limit < 0:
            raise ValueError(f"Invalid value threshold for {measurement}")

        data = df[
            ["pseudopaz", "json_id", "transm_id", "datetime", measurement]
        ].dropna(subset=[measurement]).copy()

        data["_row_index"] = data.index
        data = data.sort_values("datetime", kind="stable")

        for patient, group in data.groupby("pseudopaz", sort=False):
            rows = list(
                group[
                    ["datetime", measurement, "json_id",
                     "transm_id", "_row_index"]
                ].itertuples(index=False, name=None)
            )

            for i, (t1, v1, json1, transm1, index1) in enumerate(rows):
                for j in range(i + 1, len(rows)):
                    t2, v2, json2, transm2, index2 = rows[j]
                    time_delta = t2 - t1

                    # Later rows are even farther away in time.
                    if time_delta > time_limit:
                        break

                    value_delta = abs(v2 - v1)

                    if value_delta == 0:
                        status = "duplicate"
                    elif value_delta <= value_limit or math.isclose(
                        value_delta, value_limit, rel_tol=0, abs_tol=1e-12
                    ):
                        status = "within_tolerance"
                    else:
                        status = "conflict"

                    results.append({
                        "pseudopaz": patient,
                        "json_id": unique_ids(json1, json2),
                        "transm_id": unique_ids(transm1, transm2),
                        "measurement": measurement,
                        "values": (v1, v2),
                        "status": status,
                        "datetimes": (t1, t2),
                        "time_delta": time_delta,
                        "value_delta": value_delta,
                        "row_indices": (index1, index2),
                    })

    return pd.DataFrame(results, columns=columns)

In [ ]:
thresholds = {
    # Measurement:      (minutes, value threshold)
    "weight_kg":       (0, 0),
    "spo2_pct":        (0, 0),
    "syst_bp_mmhg":    (0, 0),
    "diast_bp_mmhg":   (0, 0),
    "map_mmhg":        (0, 0),

    # "weight_kg":       (0, 6),
    # "spo2_pct":        (0, 1.0),
    # "syst_bp_mmhg":    (0, 5.0),
    # "diast_bp_mmhg":   (0, 5.0),
    # "map_mmhg":        (0, 5.0),

    # "weight_kg":       (5, 0.2),
    # "spo2_pct":        (1, 1.0),
    # "syst_bp_mmhg":    (2, 5.0),
    # "diast_bp_mmhg":   (2, 5.0),
    # "map_mmhg":        (2, 5.0),
}

if True:
    pairs = check_measurements(df, thresholds)

    # Differences exceeding the allowed value threshold:
    conflicts = pairs.loc[pairs["status"].eq("conflict")].reset_index(drop=True)

    # Duplicates and nearby values within tolerance:
    merge_candidates = pairs.loc[
        pairs["status"].isin(["duplicate", "within_tolerance"])
    ].reset_index(drop=True)

In [ ]:
conflicts
# merge_candidates

## Heartrate values (`hr_bpm` and `hr_source`)

## Simplify the dataframe

See docstring. For example:

**`original_df`**

| pseudopaz | datetime | weight_kg | spo2_pct |
|---|---|---:|---:|
| A | 2026-01-01 09:00 | 70 | — |
| A | 2026-01-01 09:00 | — | 98 |
| B | 2026-01-01 10:00 | 80 | 96 |
| B | 2026-01-01 10:00 | 82 | 96 |

**`simplified_df`**

| pseudopaz | datetime | weight_kg | spo2_pct |
|---|---|---:|---:|
| A | 2026-01-01 09:00 | 70 | 98 |
| B | 2026-01-01 10:00 | 80 | 96 |
| B | 2026-01-01 10:00 | 82 | 96 |

In [ ]:
def simplify_measurements(df):
    '''Groups rows by patient (`pseudopaz`) and timestamp (`datetime`),
    merging each group into one row when every measurement column
    contains at most one distinct nonmissing value;
    it fills measurements from the available values
    and retains other fields from the first row.
    If any measurement has conflicting values,
    it preserves all rows in that group,
    then sorts the result by patient and timestamp and resets the index.
    '''
    measurement_cols = [
        "weight_kg",
        "spo2_pct",
        "syst_bp_mmhg",
        "diast_bp_mmhg",
        "map_mmhg",
    ]

    data = df.copy()
    rows = []

    for _, group in data.groupby(
        ["pseudopaz", "datetime"],
        sort=False,
        dropna=False,
    ):
        has_conflict = (
            group[measurement_cols]
            .nunique(dropna=True)
            .gt(1)
            .any()
        )

        if has_conflict:
            rows.extend(group.to_dict("records"))
            continue

        merged = group.iloc[0].to_dict()

        for col in measurement_cols:
            values = group[col].dropna()
            merged[col] = (
                values.iloc[0] if not values.empty else float("nan")
            )

        rows.append(merged)

    return (
        pd.DataFrame(rows, columns=data.columns)
        .sort_values(["pseudopaz", "datetime"], kind="stable")
        .reset_index(drop=True)
    )

In [ ]:
simplified_df = simplify_measurements(df)

In [ ]:
df.head()

In [ ]:
simplified_df.head()

# Export

In [ ]:
fn.export_func(df, ff.CLEANED_PROCESSED_DATA_DIR, '', filetype=[
    # 'parquet' ,
    # 'pickle' ,
    # 'excel'
])